# FairPlay AI · Buscar pares sospechosos con las reseñas

Los jugadores suelen ser los primeros en escribir *"esto es una copia de X"*. Este notebook lee reseñas públicas de Steam y arma una lista de **pares que conviene etiquetar primero**.

**Para qué sirve:** el Modelo B necesita al menos 10 pares *Posible copia*, y revisando pares al azar aparecen muy pocos. Aquí se buscan donde es más probable encontrarlos.

**Qué hace, paso a paso:**

1. Elige juegos del catálogo (al azar, con una cantidad de reseñas razonable).
2. Baja hasta 100 reseñas recientes de cada uno desde la tienda de Steam, con una pausa entre juegos.
3. Marca las reseñas con **expresiones de alerta**: *clone*, *rip-off*, *asset flip*, *stolen*, *copia*, *plagio*…
4. Mira si cerca de la alerta se nombra a **otro juego conocido** del catálogo (*"a clone of X"*). Eso da un par concreto.
5. Guarda el resultado en Drive: `MyDrive/fairplay/salidas/sospechosos_resenas.xlsx`.

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todo* y aceptar el acceso a Drive. No necesita GPU. Si se corta, vuelve a ejecutarlo: sigue donde quedó.

> ⚠️ **Es una pista, no una prueba.** *Clone* se usa mucho como género (*"un clon de Vampire Survivors muy divertido"* es un elogio). La lista dice **qué mirar**; la etiqueta la pone una persona con la [guía de etiquetado](docs/GUIA_ETIQUETADO.md).
>
> 🔒 Las reseñas son texto de usuarios de Steam: quedan en Drive y **no se suben al repositorio**.

In [ ]:
# ---- Configuración -----------------------------------------------------------
MAX_JUEGOS = 2000        # cuántos juegos revisar en total (unos 1,5 s por juego: 2000 ≈ 50 minutos)
MIN_RESENAS = 20         # con menos reseñas casi nunca hay información
MAX_RESENAS = 3000       # los juegos muy populares rara vez son la copia
PAUSA = 1.0              # segundos entre juegos, para no sobrecargar la tienda
SEMILLA = 2026           # misma semilla = mismos juegos (así se puede continuar otro día)

RAMA = "feat/clasificador-pares"
REPO = "https://github.com/sergiorebolledo/sic-ia-clone-radar"

In [ ]:
# ---- Drive, catálogo y código del proyecto -----------------------------------
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path

import pandas as pd

BASE = Path("/content/drive/MyDrive/fairplay")
CATALOGO = BASE / "salidas" / "paquete_app" / "catalogo.parquet"
CACHE = BASE / "data" / "resenas_cache.jsonl"           # reseñas ya bajadas (privado)
SALIDA = BASE / "salidas" / "sospechosos_resenas.xlsx"
assert CATALOGO.exists(), f"Falta {CATALOGO}. Corre la sección 11 del notebook principal."
CACHE.parent.mkdir(parents=True, exist_ok=True)

!rm -rf /content/app && git clone -q --depth 1 -b "$RAMA" "$REPO" /content/app
sys.path.insert(0, "/content/app/Primera fase práctica")
import resenas                                          # la lógica está en resenas.py, con tests

cat = pd.read_parquet(CATALOGO)
cat["appid"] = cat["appid"].astype(int)
nombres = dict(zip(cat["appid"], cat["name"].astype(str)))
n_resenas = dict(zip(cat["appid"], cat["n_resenas"].astype(int)))
print(f"Catálogo: {len(cat):,} juegos")

## 1. Elegir los juegos y bajar sus reseñas

Se toman juegos con entre `MIN_RESENAS` y `MAX_RESENAS` reseñas, desordenados con una semilla fija. Lo ya bajado queda en un archivo de Drive, así que cada corrida solo baja lo que falta.

In [ ]:
import json
import time

from tqdm.auto import tqdm

elegibles = cat[(cat["n_resenas"] >= MIN_RESENAS) & (cat["n_resenas"] <= MAX_RESENAS)]
juegos = elegibles.sample(frac=1, random_state=SEMILLA)["appid"].head(MAX_JUEGOS).tolist()

bajadas = {}                                            # appid -> lista de reseñas
if CACHE.exists():
    with CACHE.open(encoding="utf-8") as f:
        for linea in f:
            fila = json.loads(linea)
            bajadas[fila["appid"]] = fila["resenas"]

pendientes = [a for a in juegos if a not in bajadas]
print(f"Juegos a revisar: {len(juegos):,} · ya bajados: {len(juegos) - len(pendientes):,} · faltan: {len(pendientes):,}")

fallos = 0
with CACHE.open("a", encoding="utf-8") as f:
    for appid in tqdm(pendientes, desc="Bajando reseñas"):
        textos = resenas.descargar(appid)
        if textos is None:                              # sin respuesta: se reintenta en la próxima corrida
            fallos += 1
            if fallos >= 30:
                print("Demasiados fallos seguidos: la tienda puede estar limitando las consultas. Prueba más tarde.")
                break
            continue
        fallos = 0
        bajadas[appid] = textos
        f.write(json.dumps({"appid": appid, "resenas": textos}, ensure_ascii=False) + "\n")
        f.flush()
        time.sleep(PAUSA)
print(f"Reseñas disponibles de {sum(a in bajadas for a in juegos):,} juegos")

## 2. Buscar alertas y menciones

Para cada juego se cuenta cuántas reseñas tienen una expresión de alerta y qué otros juegos **conocidos** se nombran cerca de ella. Solo cuentan como mención los títulos distintivos (ver `Titulos` en `resenas.py`): muchos juegos se llaman como palabras comunes.

In [ ]:
titulos = resenas.Titulos(nombres, n_resenas)
filas = [resenas.analizar(a, bajadas[a], titulos) for a in juegos if a in bajadas]
res = pd.DataFrame(filas)
res["nombre"] = res["appid"].map(nombres)
res["url"] = "https://store.steampowered.com/app/" + res["appid"].astype(str)

print(f"Juegos analizados: {len(res):,}")
print(f"  con alguna reseña de alerta: {(res['n_alerta'] > 0).sum():,}")
print(f"  que además nombran a otro juego: {(res['menciones'].map(len) > 0).sum():,}")

## 3. Pares para revisar

Cada fila es un par: el **juego reseñado** (A) y el **juego que nombran sus reseñas** (B). Arriba quedan los pares con más reseñas que los relacionan.

In [ ]:
pares = []
for r in res.itertuples():
    for b, n in r.menciones.items():
        pares.append({"juego_a": r.nombre, "juego_b": nombres[b], "resenas_que_lo_dicen": n,
                      "resenas_con_alerta": r.n_alerta, "resenas_leidas": r.n_resenas,
                      "ejemplo": r.ejemplo, "appid_a": r.appid, "appid_b": b,
                      "url_a": r.url, "url_b": f"https://store.steampowered.com/app/{b}"})
pares = pd.DataFrame(pares)
if not pares.empty:
    pares = pares.sort_values(["resenas_que_lo_dicen", "resenas_con_alerta"], ascending=False).reset_index(drop=True)

juegos_alerta = (res[res["n_alerta"] > 0]
                 .sort_values(["n_alerta", "frac_alerta"], ascending=False)
                 [["nombre", "n_alerta", "n_resenas", "frac_alerta", "ejemplo", "appid", "url"]]
                 .reset_index(drop=True))

with pd.ExcelWriter(SALIDA) as w:
    pares.to_excel(w, sheet_name="Pares", index=False)
    juegos_alerta.to_excel(w, sheet_name="Juegos con alertas", index=False)
print("Guardado en", SALIDA)

pd.set_option("display.max_colwidth", 140)
pares.head(30)[["juego_a", "juego_b", "resenas_que_lo_dicen", "ejemplo"]] if not pares.empty else "Sin pares todavía: sube MAX_JUEGOS."

## 4. Qué hacer con la lista

1. Abre `sospechosos_resenas.xlsx` en Drive, hoja **Pares**.
2. Para cada par, abre los dos enlaces de Steam y compara descripción, imágenes y capturas.
3. Etiqueta el par en la app publicada, pestaña **Comparar dos juegos**: escribe los dos nombres, pulsa *Comparar* y elige *Posible copia*, *Similar* o *No similar*.
4. La hoja **Juegos con alertas** trae juegos cuyas reseñas hablan de copia sin nombrar a otro juego del catálogo (por ejemplo, copias de juegos de consola o de celular). Sirven para buscarlos en la pestaña *Buscar candidatos*.

**Cómo leer los resultados:**

* Muchos pares serán *Similar*: la reseña dice "clon de X" para describir el género. Eso también es útil: son los negativos difíciles.
* Si un par tiene varias reseñas que lo dicen y el ejemplo habla de arte, capturas o texto reutilizado, es buen candidato a *Posible copia*.
* Estos pares se eligieron **por sospechosos**, no al azar. Sirven para que el Modelo B vea ejemplos de copia, pero la evaluación del informe se apoya en el lote del notebook principal, que mezcla pares de todo tipo.

**Límites de este método:** solo lee hasta 100 reseñas recientes por juego, entiende expresiones en inglés y español, y no ve juegos sin reseñas (muchas copias no tienen).